In [4]:
import tensorflow as tf
import tensorflow_transform as tft
import keras

print('Tf version: ', tf.__version__)
print('TFT version: ', tft.__version__)
print('Keras version: ', keras.__version__)

Tf version:  2.17.0
TFT version:  1.14.0
Keras version:  3.10.0


In [7]:
model = keras.models.load_model("my_laptop_prediction.keras")

In [8]:
raw_features = {
    'Company':          tf.constant([b'Lenovo']),
    'TypeName':         tf.constant([b'Ultrabook']),
    'Inches':           tf.constant([15.6]),
    'ScreenResolution': tf.constant([b'1920x1080']),
    'Cpu':              tf.constant([b'Intel Core i7']),
    'Ram':              tf.constant([16.0]),
    'Memory':           tf.constant([b'512.0']),
    'Gpu':              tf.constant([b'NVIDIA GTX 1650']),
    'OpSys':            tf.constant([b'Windows 10']),
    'Weight':           tf.constant([2.5])
}


In [9]:
tft_output = tft.TFTransformOutput('train')
transformed_features = tft_output.transform_raw_features(raw_features)
print(transformed_features)


INFO:tensorflow:struct2tensor is not available.
INFO:tensorflow:tensorflow_decision_forests is not available.
INFO:tensorflow:tensorflow_text is not available.
{'is_intel_iris_gpu': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([0], dtype=int64)>, 'is_intel_cpu': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([1], dtype=int64)>, 'is_i5_cpu': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([0], dtype=int64)>, 'is_nvidia_geforce_gpu': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([0], dtype=int64)>, 'is_other_gpu': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([1], dtype=int64)>, 'is_i3_cpu': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([0], dtype=int64)>, 'is_amd_cpu': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([0], dtype=int64)>, 'is_general_cpu': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([0], dtype=int64)>, 'company_xf': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([0], dtype=int64)>, 'is_nvidia_quadro_gpu': <tf.Tensor: shape=(1,), dtype=int64, numpy=

In [13]:
ordered_keys = list(transformed_features.keys())  # careful: dict order must match training
features_list = [
    tf.cast(tf.expand_dims(transformed_features[key], axis=-1), tf.float32) 
    for key in ordered_keys
]

# Concatenate along axis=1 to get shape (1, num_features)
input_vector = tf.concat(features_list, axis=1)

print(input_vector.shape)  # (1, 32)

(1, 32)


In [15]:
pred_scaled = model.predict(input_vector)
print(pred_scaled)  # your model's price prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
[[-158.82083]]


In [16]:
import pandas as pd

df = pd.read_csv('dataset/train_cleaned.csv')
MEAN_PRICE = df['Price'].mean()
STD_PRICE = df['Price'].std()

raw_price_pred = pred_scaled * STD_PRICE + MEAN_PRICE
raw_price_pred

array([[-6170081.]], dtype=float32)